## Name:Ankita Subhash Patil
## Task 3: Chatbot using Transformers

In [ ]:
# Install Hugging Face Transformers and PyTorch
!pip install transformers torch --quiet
print("✅ Installation complete!")

✅ Installation complete!


In [ ]:
# Import Hugging Face pipeline — high-level API for model inference
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')  # Suppress non-critical warnings

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [ ]:
# Load the text2text-generation pipeline using FLAN-T5
# text2text-generation: takes an instruction/question → generates a relevant answer
print("📥 Loading model: google/flan-t5-base")
print("   (First run downloads the model — may take ~1-2 minutes)\n")

chatbot_pipeline = pipeline(
    task="text-generation",   # Instruction-to-response task
    model="google/flan-t5-base",   # Pre-trained model from Hugging Face Hub
    max_new_tokens=200             # Max tokens to generate per response
)

print("✅ Model loaded successfully!")
print("   • Model  : google/flan-t5-base")
print("   • Task   : text-generation")
print("   • Source : https://huggingface.co/google/flan-t5-base")

📥 Loading model: google/flan-t5-base
   (First run downloads the model — may take ~1-2 minutes)



config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

✅ Model loaded successfully!
   • Model  : google/flan-t5-base
   • Task   : text-generation
   • Source : https://huggingface.co/google/flan-t5-base


In [ ]:
def generate_response(user_input, conversation_history=None):
    """
    Generate a chatbot response using the FLAN-T5 transformer model.

    Formats the user input as a prompt (with optional conversation context),
    runs it through the Hugging Face pipeline, and returns the model's reply.

    Args:
        user_input           (str)  : The message typed by the user.
        conversation_history (list) : List of past (user_msg, bot_msg) tuples
                                      to provide context to the model.

    Returns:
        response (str) : The chatbot's generated reply.
    """

    # -------------------------------------------------------
    # Step A: Build the prompt
    # FLAN-T5 is instruction-tuned — we frame input as a clear
    # instruction so the model knows how to respond.
    # -------------------------------------------------------

    if conversation_history and len(conversation_history) > 0:
        # Include last 3 turns for multi-turn context
        recent_history = conversation_history[-3:]
        context = ""
        for past_user, past_bot in recent_history:
            context += f"User: {past_user}\nAssistant: {past_bot}\n"

        # Prompt includes conversation history for contextual awareness
        prompt = (
            f"You are a helpful AI assistant. Continue this conversation accurately.\n\n"
            f"{context}"
            f"User: {user_input}\n"
            f"Assistant:"
        )
    else:
        # First turn — no prior history, simple instruction prompt
        prompt = (
            f"You are a helpful AI assistant. Answer the following question clearly and accurately.\n"
            f"User: {user_input}\n"
            f"Assistant:"
        )

    # -------------------------------------------------------
    # Step B: Generate response using the transformer pipeline
    # -------------------------------------------------------
    output = chatbot_pipeline(
        prompt,
        max_new_tokens=200,   # Limit output length
        do_sample=True,       # Enable sampling for natural response variety
        temperature=0.7,      # Balanced creativity vs factual accuracy
        top_p=0.9             # Nucleus sampling — top 90% probability mass
    )

    # -------------------------------------------------------
    # Step C: Extract the generated text from pipeline output
    # -------------------------------------------------------
    response = output[0]['generated_text'].strip()

    # Fallback if model returns empty or whitespace response
    if not response:
        response = "I'm not sure about that. Could you please rephrase your question?"

    return response


print("✅ generate_response() function defined and ready.")

✅ generate_response() function defined and ready.


In [ ]:
def run_chatbot():
    """
    Starts the interactive chatbot loop.

    - Accepts real user input from the console
    - Maintains full conversation history across turns
    - Generates responses using the FLAN-T5 transformer model
    - Handles blank input and errors gracefully
    - Exits cleanly when user types 'exit' or 'quit'
    """

    print("=" * 60)
    print("   🤖 AI Chatbot — Powered by Hugging Face Transformers")
    print("=" * 60)
    print("   Model : google/flan-t5-base")
    print("   Type 'exit' or 'quit' to end the conversation.")
    print("=" * 60 + "\n")

    # Opening greeting message
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

    # Stores all (user_message, bot_response) pairs for conversation context
    conversation_history = []

    # Main loop — keeps running until user types 'exit' or 'quit'
    while True:

        # Read user input from console
        user_input = input("You: ").strip()

        # Handle empty input — prompt user to type something
        if not user_input:
            print("Chatbot: I didn't catch that. Could you please say something?\n")
            continue

        # Exit condition — end chatbot gracefully
        if user_input.lower() in ["exit", "quit"]:
            print("\nChatbot: Thank you for chatting with me! Goodbye! 👋")
            print("=" * 60)
            break

        # Generate response using transformer model
        try:
            response = generate_response(user_input, conversation_history)
        except Exception as e:
            # Gracefully handle unexpected model errors
            response = "Sorry, something went wrong. Please try asking again!"
            print(f"   [Debug Error: {e}]")

        # Display the chatbot reply
        print(f"\nChatbot: {response}\n")

        # Append this turn to history for future contextual responses
        conversation_history.append((user_input, response))


# ▶️ Start the interactive chatbot
run_chatbot()

   🤖 AI Chatbot — Powered by Hugging Face Transformers
   Model : google/flan-t5-base
   Type 'exit' or 'quit' to end the conversation.

Chatbot: Hello! I am your AI assistant. How can I help you today?



Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Chatbot: You are a helpful AI assistant. Answer the following question clearly and accurately.
User: Hello
Assistant:pril after taking it off by being fired Monday Night while waiting around from home following indictation Sunday morning as reports started and thousands to come after Rochester refused $560,000 from state workers Tuesday who had promised $840,300 in return back pay due this September amid the ongoing state police probe uphol tics." Dr Amidran said we would continue to hold services of humanitarian charity from here as all government funds such we could pay these costs so please spare Dr May and Senator DeSachilles until there were appropriate channels or sources



Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Chatbot: You are a helpful AI assistant. Continue this conversation accurately.

User: Hello
Assistant: You are a helpful AI assistant. Answer the following question clearly and accurately.
User: Hello
Assistant:pril after taking it off by being fired Monday Night while waiting around from home following indictation Sunday morning as reports started and thousands to come after Rochester refused $560,000 from state workers Tuesday who had promised $840,300 in return back pay due this September amid the ongoing state police probe uphol tics." Dr Amidran said we would continue to hold services of humanitarian charity from here as all government funds such we could pay these costs so please spare Dr May and Senator DeSachilles until there were appropriate channels or sources
User: What is ai
Assistant:



In [ ]:
def demo_chatbot(test_inputs):
    """
    Simulates a chatbot conversation using predefined user messages.
    Displays full conversation output statically inside the notebook.

    Args:
        test_inputs (list): List of predefined user messages to test.
    """

    print("\n" + "=" * 60)
    print("       📋 Chatbot Demo — Sample Conversation")
    print("=" * 60)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

    conversation_history = []  # Start with empty conversation history

    for user_msg in test_inputs:

        # Exit keyword — end demo gracefully
        if user_msg.lower() in ["exit", "quit"]:
            print(f"You: {user_msg}")
            print("\nChatbot: Thank you for chatting with me! Goodbye! 👋")
            print("=" * 60)
            break

        # Generate response from transformer model
        response = generate_response(user_msg, conversation_history)

        # Fallback for empty model output
        if not response.strip():
            response = "I'm not sure how to respond to that. Could you rephrase?"

        # Display this conversation turn
        print(f"You: {user_msg}")
        print(f"Chatbot: {response}\n")

        # Save turn to history for contextual follow-up questions
        conversation_history.append((user_msg, response))


# ============================================================
# Test inputs — matches the assignment's Expected Chatbot Flow
# ============================================================
sample_inputs = [
    "Hello",
    "What is Artificial Intelligence?",
    "Who created Python?",
    "What is deep learning?",
    "What is Natural Language Processing?",
    "What is a transformer model?",
    "Thank you",
    "exit"
]

demo_chatbot(sample_inputs)